# # Chicago Taxi Trips — Week of 11–17 September 2023

*Summary placeholder — written after the analysis is complete.*

# ## 1. Data loading
#
# All figures describe **reported** trips (the City notes that not every trip is reported), not total market volume.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

os.makedirs("figures/analysis", exist_ok=True)

# Chart styling: one blue for magnitude, orange to flag weekends, muted grey for context
BLUE, ORANGE, GREY = "#2a78d6", "#eb6834", "#9a9a95"
plt.rcParams.update({
    "figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": "#e6e6e3", "grid.linewidth": 0.8,
    "axes.axisbelow": True, "axes.titlesize": 13, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "axes.labelsize": 10.5,
})

raw = pd.read_csv("trips.csv", dtype={"pickup_census_tract": "string", "dropoff_census_tract": "string"})
print("shape:", raw.shape)
print()
print(raw.dtypes)

shape: (135768, 23)

trip_id                           str
taxi_id                           str
trip_start_timestamp              str
trip_end_timestamp                str
trip_seconds                  float64
trip_miles                    float64
pickup_census_tract            string
dropoff_census_tract           string
pickup_community_area         float64
dropoff_community_area        float64
fare                          float64
tips                          float64
tolls                         float64
extras                        float64
trip_total                    float64
payment_type                      str
company                           str
pickup_centroid_latitude      float64
pickup_centroid_longitude     float64
pickup_centroid_location          str
dropoff_centroid_latitude     float64
dropoff_centroid_longitude    float64
dropoff_centroid_location         str
dtype: object


In [2]:
# Missing values per column, and uniqueness of trip_id
miss = raw.isna().sum().to_frame("n_missing")
miss["pct_missing"] = miss["n_missing"] / len(raw) * 100
print(miss[miss.n_missing > 0])
print()
print("duplicate trip_id:", raw.trip_id.duplicated().sum(), "| fully duplicated rows:", raw.duplicated().sum())
print("start range:", raw.trip_start_timestamp.min(), "->", raw.trip_start_timestamp.max())

                            n_missing  pct_missing
trip_end_timestamp                  1         0.00
trip_seconds                       33         0.02
pickup_census_tract             71404        52.59
dropoff_census_tract            73842        54.39
pickup_community_area            3815         2.81
dropoff_community_area          13794        10.16
fare                              324         0.24
tips                              324         0.24
tolls                             324         0.24
extras                            324         0.24
trip_total                        324         0.24
pickup_centroid_latitude         3775         2.78
pickup_centroid_longitude        3775         2.78
pickup_centroid_location         3775         2.78
dropoff_centroid_latitude       12872         9.48
dropoff_centroid_longitude      12872         9.48
dropoff_centroid_location       12872         9.48

duplicate trip_id: 0 | fully duplicated rows: 0
start range: 2023-09-11T00:00:00.

# ## 2. Data quality checks and cleaning
#
# First inspect, then decide. Nothing is dropped until the decisions are listed in the table at the end of this section.

In [3]:
df = raw.copy()
df["start"] = pd.to_datetime(df.trip_start_timestamp)
df["end"] = pd.to_datetime(df.trip_end_timestamp)

num = ["trip_seconds", "trip_miles", "fare", "tips", "tolls", "extras", "trip_total"]
q = df[num].describe(percentiles=[.01, .5, .99, .999]).T
print(q)
print()
print("Zero counts / negative counts:")
print(pd.DataFrame({"zero": (df[num] == 0).sum(), "negative": (df[num] < 0).sum()}))

                  count     mean      std  min   1%      50%      99%     99.9%       max
trip_seconds 135,735.00 1,346.54 1,762.25 0.00 0.00 1,020.00 4,872.98 12,416.09 86,160.00
trip_miles   135,768.00     6.94     7.85 0.00 0.00     3.24    27.14     44.08    792.90
fare         135,444.00    23.87    19.60 0.00 3.25    17.75    75.50    150.00    911.00
tips         135,444.00     3.38     4.67 0.00 0.00     2.00    17.10     33.33    130.00
tolls        135,444.00     0.13    20.42 0.00 0.00     0.00     0.00      5.00  4,444.44
extras       135,444.00     2.54    18.31 0.00 0.00     0.00    35.50     65.75  4,445.44
trip_total   135,444.00    30.12    43.93 0.00 3.25    21.25   104.43    186.53  8,901.13

Zero counts / negative counts:
                zero  negative
trip_seconds    2731         0
trip_miles     13993         0
fare             135         0
tips           60991         0
tolls         134709         0
extras         80677         0
trip_total       134         0


In [4]:
print(df.payment_type.value_counts(dropna=False))
print()
print(df.company.value_counts(dropna=False).head(25))
print("n companies:", df.company.nunique(), "| n taxis:", df.taxi_id.nunique(), "| taxi_id missing:", df.taxi_id.isna().sum())

payment_type
Credit Card    60291
Cash           40053
Mobile         18594
Prcard         11961
Unknown         4784
No Charge         54
Dispute           31
Name: count, dtype: int64

company
Flash Cab                               25950
Taxi Affiliation Services               25727
Sun Taxi                                15122
Taxicab Insurance Agency Llc            13914
City Service                            13747
Chicago Independents                     8004
5 Star Taxi                              6121
Medallion Leasin                         5003
Globe Taxi                               4772
Blue Ribbon Taxi Association             4390
Star North Taxi Management Llc           2815
Taxicab Insurance Agency, LLC            2379
Choice Taxi Association                  2022
Chicago City Taxi Association            1386
Choice Taxi Association Inc              1064
U Taxicab                                1048
Chicago Taxicab                           406
Patriot Taxi Dba Peace 

In [5]:
# Does trip_total equal fare + tips + tolls + extras?
parts = df[["fare", "tips", "tolls", "extras"]].sum(axis=1, min_count=4)
df["total_gap"] = df.trip_total - parts
g = df.total_gap.dropna()
print("rows with computable gap:", len(g))
print("exact match (|gap|<0.005):", (g.abs() < 0.005).sum(), f"({(g.abs() < 0.005).mean():.2%})")
print("gap quantiles:")
print(g.quantile([0, .001, .01, .05, .5, .95, .99, .999, 1]))
print()
print("Gap value counts (rounded to 0.01), top 12:")
print(g.round(2).value_counts().head(12))

rows with computable gap: 135444
exact match (|gap|<0.005): 87247 (64.42%)
gap quantiles:
0.00   -0.00
0.00   -0.00
0.01    0.00
0.05    0.00
0.50    0.00
0.95    0.50
0.99    2.50
1.00    3.00
1.00    3.00
Name: total_gap, dtype: float64

Gap value counts (rounded to 0.01), top 12:
total_gap
0.00    87247
0.50    46522
3.00      804
2.50      751
1.00      109
1.95        5
1.50        4
2.00        2
Name: count, dtype: int64
